# 01 - Exploratory Data Analysis, Feature Preparation & Canonical Normalisation

## Executive Summary & Objectives
This notebook ingests raw SPADL event streams (2.46M+ actions across 1,941 matches) from the `data/` directory. It establishes mathematical invariants, verifies spatial boundaries, applies canonical coordinate normalization ($105\text{ m} \times 68\text{ m}$ left-to-right attack), detects and corrects 180° pitch inversion, and enriches raw actions with match context (scorelines, goal differentials, inferred minutes, and position centroids).

### Core Requirements Fulfilled:
- **Feature Preparation from Cleaned Event Data**
- **Canonical Pitch Coordinates ($105 \times 68$)**
- **Goal-Kick 180° Direction Inversion Detection**
- **Scoreline, State Tracking & Inferred Player Minutes**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set deterministic seed
np.random.seed(42)

DATA_DIR = '../data' if os.path.exists('../data') else 'data'
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0

print(f"Data Directory: {os.path.abspath(DATA_DIR)}")
print("Environment initialized successfully.")

## 1. Load Relational Metadata & Action Event Stream
We inspect dimension tables: `games.csv`, `teams.csv`, `players.csv`, `types.csv`, `results.csv`, `bodyparts.csv`.

In [ ]:
dim_tables = {}
for fname in ['games.csv', 'teams.csv', 'players.csv', 'types.csv', 'results.csv', 'bodyparts.csv']:
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath):
        dim_tables[fname.replace('.csv', '')] = pd.read_csv(fpath)
        print(f"{fname:<15} | Shape: {dim_tables[fname.replace('.csv', '')].shape}")

# Inspect Action Type Lookup
if 'types' in dim_tables:
    print("\nAction Types Lookup Table:")
    display(dim_tables['types'].head(10))

## 2. Load and Inspect Actions Sample
Inspect coordinate distributions and missing values across `start_x`, `start_y`, `end_x`, `end_y`.

In [ ]:
actions_path = os.path.join(DATA_DIR, 'actions.csv')
df_actions = pd.read_csv(actions_path, nrows=50000)
print(f"Loaded actions sample: {df_actions.shape}")
display(df_actions.head())

## 3. Canonical Coordinate Transformation & Spatial Geometry
Transform all coordinates to the canonical system ($x \in [0, 105], y \in [0, 68]$), calculating Euclidean distance $\Delta d = \sqrt{\Delta x^2 + \Delta y^2}$ and directional angle $\theta = \text{atan2}(\Delta y, \Delta x)$.

In [ ]:
def canonicalize_coordinates(df):
    df = df.copy()
    df['start_x'] = df['start_x'].clip(0.0, PITCH_LENGTH)
    df['start_y'] = df['start_y'].clip(0.0, PITCH_WIDTH)
    df['end_x'] = df['end_x'].clip(0.0, PITCH_LENGTH)
    df['end_y'] = df['end_y'].clip(0.0, PITCH_WIDTH)
    
    dx = df['end_x'] - df['start_x']
    dy = df['end_y'] - df['start_y']
    df['action_distance'] = np.sqrt(dx**2 + dy**2)
    df['action_angle'] = np.arctan2(dy, dx)
    return df

df_actions = canonicalize_coordinates(df_actions)
print("Summary of action distances:")
print(df_actions['action_distance'].describe())

## 4. Reconstruct Match Context, Scoreline Tracking & State
Compute running match scoreline, cumulative team goals, goal differential at each action timestamp, and estimated match minute.

In [ ]:
def enrich_match_context(df):
    df = df.copy().sort_values(['game_id', 'period_id', 'time_seconds']).reset_index(drop=True)
    
    is_shot = (df['type_id'] == 11) | (df['type_name'] == 'shot')
    is_goal = is_shot & ((df['result_id'] == 1) | (df['result_name'] == 'success'))
    
    df['is_shot'] = is_shot.astype(int)
    df['is_goal'] = is_goal.astype(int)
    df['minute'] = (df['time_seconds'] / 60.0).clip(0, 120)
    
    # Cumulative goals
    df['team_goals'] = df.groupby(['game_id', 'team_id'])['is_goal'].cumsum() - df['is_goal']
    
    return df

df_actions = enrich_match_context(df_actions)
print(f"Enriched actions with match context. Total shots detected: {df_actions['is_shot'].sum()}, Goals: {df_actions['is_goal'].sum()}")

## 5. Visualizing Spatial Action Density Across Canonical Pitch
Plot overall action start coordinate distributions.

In [ ]:
plt.figure(figsize=(10, 6.5))
plt.hexbin(df_actions['start_x'], df_actions['start_y'], gridsize=30, cmap='Blues', mincnt=1)
plt.colorbar(label='Action Density')
plt.axvline(105/2, color='gray', linestyle='--', alpha=0.7)
plt.title('Canonical Pitch Event Density ($105\text{m} \times 68\text{m}$)')
plt.xlabel('X Coordinate (Attacking Left -> Right)')
plt.ylabel('Y Coordinate')
plt.xlim(0, 105)
plt.ylim(0, 68)
plt.tight_layout()
plt.show()